<a href="https://colab.research.google.com/github/natdanaiii/Trading/blob/main/Grid_trading_V0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# V0-B Fixed Grid + Monthly DCA Benchmark

V0-B trading logic remains frozen. Monthly DCA is evaluation-only: same 3,000 USDT starting capital, 24 monthly buys over Jan-2024 to Dec-2025 (125 USDT/month), first 1-minute candle on day 1 of each month, 0.1% buy fee, no final sale. Final DCA portfolio = remaining cash + BTC × final close.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from google.colab import userdata
import base64,bisect,heapq,json,os,time
from datetime import datetime,timezone
import numpy as np,pandas as pd,requests

SYMBOL="BTCUSDT"; INITIAL_CAPITAL=3000.0
GRID_FLOOR=38000.0; GRID_CEILING=127000.0; GRID_GAP=1000.0
BUY_FEE=SELL_FEE=0.001
TIMEFRAME="1m"; START_DATE="2024-01-01"; END_DATE="2026-01-01"
DATA_DIR="/content/drive/MyDrive/03.Trading/00.Live Trading"


# 1. Frozen V0-B Trading System


In [ ]:
def build_grid():
    b=np.arange(GRID_FLOOR,GRID_CEILING,GRID_GAP,dtype=float)
    return pd.DataFrame({"grid_id":np.arange(1,len(b)+1),"buy_price":b,"sell_target":b+GRID_GAP})

def derive_init(g,p0):
    seed=g.sell_target>p0; reserve=~seed
    weight=int(reserve.sum())+float(np.sum(p0/g.loc[seed,"buy_price"].to_numpy(float)))
    q=INITIAL_CAPITAL/weight
    g=g.copy(); g["seed_at_start"]=seed; g["order_size_usdt"]=q
    g["normal_net_btc"]=q/g.buy_price*(1-BUY_FEE)
    g["initial_cost"]=np.where(seed,g.normal_net_btc/(1-BUY_FEE)*p0,0.0)
    reserved=float(reserve.sum()*q); seeded=float(g.initial_cost.sum())
    assert np.isclose(reserved+seeded,INITIAL_CAPITAL,atol=1e-8)
    return g,{"normal_order_size_usdt":float(q),"funding_weight":float(weight),
              "initial_sell_positions":int(seed.sum()),"initial_buy_levels":int(reserve.sum()),
              "reserved_cash_usdt":reserved,"seeded_btc_cost_usdt":seeded}

def make_pos(tid,idx,t,market_buy,grid_buy,target,q,portfolio,kind):
    if kind=="INITIAL_SEED":
        net=q/grid_buy*(1-BUY_FEE); gross=net/(1-BUY_FEE); cost=gross*market_buy
    else:
        cost=q; gross=cost/market_buy
    fee_btc=gross*BUY_FEE; btc=gross-fee_btc
    gross_sell=btc*target; sell_fee=gross_sell*SELL_FEE
    return {"trade_id":tid,"grid_index":idx,"entry_type":kind,"status":"OPEN","buy_time":t,
            "buy_price":float(market_buy),"sell_target":float(target),"sell_time":pd.NaT,
            "order_size_usdt":float(cost),"portfolio_value_at_buy":float(portfolio),
            "order_pct_of_portfolio":float(cost/portfolio),"portfolio_value_at_sell":np.nan,
            "btc_amount":float(btc),"buy_fee_btc":float(fee_btc),"sell_fee_usdt":float(sell_fee),
            "net_sell_usdt":float(gross_sell-sell_fee),"net_pnl":np.nan}

def run_v0(data):
    data=data.sort_values("open_time").reset_index(drop=True)
    p0=float(data.iloc[0].open); t0=data.iloc[0].open_time
    g,init=derive_init(build_grid(),p0); q=init["normal_order_size_usdt"]
    cash=INITIAL_CAPITAL; btc=0.0; pos={}; active={}; heap=[]; ev=[]; tid=eid=0; buy_fee_total=0.0
    for idx in g.index[g.seed_at_start]:
        r=g.loc[idx]; tid+=1; pv=cash+btc*p0
        p=make_pos(tid,idx,t0,p0,float(r.buy_price),float(r.sell_target),q,pv,"INITIAL_SEED")
        cb,bb=cash,btc; cash-=p["order_size_usdt"]; btc+=p["btc_amount"]; buy_fee_total+=p["buy_fee_btc"]*p0
        pos[tid]=p; active[idx]=tid; heapq.heappush(heap,(p["sell_target"],tid)); eid+=1
        ev.append({"event_id":eid,"time":t0,"side":"BUY","trade_id":tid,"grid_index":idx,
                   "cash_movement":-p["order_size_usdt"],"grid_cashflow":0.0,"cash_before":cb,
                   "cash_after":cash,"btc_before":bb,"btc_after":btc,"initialization_trade":True})
    init={**init,"start_time":t0,"start_price":p0,"initial_cash":float(cash),"initial_btc":float(btc),
          "initial_btc_cost_usdt":init["seeded_btc_cost_usdt"],"initial_buy_fee_usdt":float(buy_fee_total),
          "initial_btc_allocation_pct":float(init["seeded_btc_cost_usdt"]/INITIAL_CAPITAL*100)}

    buys=g.buy_price.to_numpy(float); targets=g.sell_target.to_numpy(float); L=buys.tolist()
    eq=np.empty(len(data)); cash_c=np.empty(len(data)); btc_c=np.empty(len(data))
    prev=None; realized=sell_fee_total=0.0; cycles=0; tol=1e-12
    for i,c in enumerate(data.itertuples(index=False)):
        t=c.open_time; o,h,l,cl=map(float,(c.open,c.high,c.low,c.close))
        cash_start=cash; sold=set()
        while heap and heap[0][0]<=h+tol:
            _,x=heapq.heappop(heap); p=pos[x]
            if p["status"]!="OPEN": continue
            idx=p["grid_index"]; cb,bb=cash,btc; pv=cb+bb*p["sell_target"]
            cash+=p["net_sell_usdt"]; btc-=p["btc_amount"]
            if abs(btc)<1e-12: btc=0.0
            pnl=p["net_sell_usdt"]-p["order_size_usdt"]
            p.update(status="CLOSED",sell_time=t,portfolio_value_at_sell=pv,net_pnl=float(pnl))
            active.pop(idx,None); sold.add(idx); realized+=pnl; sell_fee_total+=p["sell_fee_usdt"]; cycles+=1; eid+=1
            ev.append({"event_id":eid,"time":t,"side":"SELL","trade_id":x,"grid_index":idx,
                       "cash_movement":p["net_sell_usdt"],"grid_cashflow":pnl,"cash_before":cb,
                       "cash_after":cash,"btc_before":bb,"btc_after":btc,"initialization_trade":False})
        budget=cash_start; start=o if prev is None else max(prev,o)
        if l<start:
            a=bisect.bisect_left(L,l); b=bisect.bisect_left(L,start)
            for idx in range(b-1,a-1,-1):
                if idx in active or idx in sold: continue
                if budget+tol<q: break
                bp=float(buys[idx]); target=float(targets[idx]); cb,bb=cash,btc; pv=cb+bb*bp; tid+=1
                p=make_pos(tid,idx,t,bp,bp,target,q,pv,"GRID_BUY")
                budget-=q; cash-=q; btc+=p["btc_amount"]; buy_fee_total+=p["buy_fee_btc"]*bp
                pos[tid]=p; active[idx]=tid; heapq.heappush(heap,(target,tid)); eid+=1
                ev.append({"event_id":eid,"time":t,"side":"BUY","trade_id":tid,"grid_index":idx,
                           "cash_movement":-q,"grid_cashflow":0.0,"cash_before":cb,"cash_after":cash,
                           "btc_before":bb,"btc_after":btc,"initialization_trade":False})
        eq[i]=cash+btc*cl; cash_c[i]=cash; btc_c[i]=btc; prev=cl
    curve=pd.DataFrame({"open_time":data.open_time,"close":data.close,"cash":cash_c,"btc":btc_c,"equity":eq})
    return {"data":data,"grid":g,"initialization":init,"positions":pos,"events":pd.DataFrame(ev),
            "equity_curve":curve,"final_cash":float(cash),"final_btc":float(btc),
            "realized_profit":float(realized),"completed_cycles":int(cycles),
            "total_buy_fee_usdt":float(buy_fee_total),"total_sell_fee_usdt":float(sell_fee_total)}


# 2. Backtest + Monthly DCA Benchmark


In [ ]:
def load_data():
    f=os.path.join(DATA_DIR,f"{SYMBOL}-{TIMEFRAME}-combined.csv"); d=pd.read_csv(f)
    d["open_time"]=pd.to_datetime(d.open_time,utc=True)
    d[["open","high","low","close","volume"]]=d[["open","high","low","close","volume"]].astype(float)
    s=pd.Timestamp(START_DATE,tz="UTC"); e=pd.Timestamp(END_DATE,tz="UTC")
    return d.drop_duplicates("open_time").sort_values("open_time").loc[lambda x:(x.open_time>=s)&(x.open_time<e)].reset_index(drop=True)

def perf(data,curve):
    e=curve.equity.to_numpy(float); peak=np.maximum.accumulate(e); dd=e/peak-1
    final=float(e[-1]); ret=final/INITIAL_CAPITAL-1
    days=(data.open_time.iloc[-1]-data.open_time.iloc[0]).total_seconds()/86400
    ann=float(np.expm1(np.log(final/INITIAL_CAPITAL)*(365.25/days)))
    mdd=float(dd.min()); cal=float(ann/abs(mdd)) if mdd<0 else np.nan
    out=curve.copy(); out["drawdown"]=dd
    return {"final_equity":final,"net_return":ret,"annualized_return":ann,
            "max_drawdown":mdd,"calmar_ratio":cal,"equity_curve":out}

def monthly_dca(data):
    w=data.reset_index(drop=True).copy(); w["month"]=w.open_time.dt.strftime("%Y-%m")
    day1=w.loc[w.open_time.dt.day.eq(1)]; ix=day1.groupby("month",sort=True).head(1).index.to_numpy()
    months=w.month.drop_duplicates().tolist(); found=w.loc[ix,"month"].tolist()
    missing=sorted(set(months)-set(found))
    if missing: raise ValueError("Missing first-day DCA candle: "+",".join(missing))
    n=len(ix); amount=INITIAL_CAPITAL/n; px=w.loc[ix,"open"].to_numpy(float)
    gross=amount/px; fee_btc=gross*BUY_FEE; net=gross-fee_btc; fee_usdt=fee_btc*px
    add=np.zeros(len(w)); spend=np.zeros(len(w)); add[ix]=net; spend[ix]=amount
    btc=np.cumsum(add); cash=INITIAL_CAPITAL-np.cumsum(spend); cash[np.abs(cash)<1e-10]=0
    curve=pd.DataFrame({"open_time":w.open_time,"close":w.close,"cash":cash,"btc":btc,"equity":cash+btc*w.close.to_numpy(float)})
    s=perf(w,curve); schedule=pd.DataFrame({"month":w.loc[ix,"month"].to_numpy(),"buy_time":w.loc[ix,"open_time"].to_numpy(),
        "buy_price":px,"dca_amount_usdt":amount,"buy_fee_usdt_equiv":fee_usdt,"net_btc":net})
    return {"number_of_buys":int(n),"dca_amount_usdt":float(amount),"total_invested_usdt":float(amount*n),
            "total_buy_fee_usdt_equiv":float(fee_usdt.sum()),"final_cash":float(cash[-1]),"final_btc":float(btc[-1]),
            "first_buy_time":schedule.buy_time.iloc[0],"last_buy_time":schedule.buy_time.iloc[-1],"schedule":schedule,**s}

def history(r):
    fc=float(r["data"].iloc[-1].close); ft=r["data"].iloc[-1].open_time; rows=[]
    for tid in sorted(r["positions"]):
        p=r["positions"][tid]; closed=p["status"]=="CLOSED"
        rows.append({"Trade ID":tid,"Status":p["status"],"Buy Time":p["buy_time"],"Buy Price":p["buy_price"],
            "Sell Target":p["sell_target"],"Sell Time":p["sell_time"] if closed else pd.NaT,
            "Order Size (USDT)":p["order_size_usdt"],"Net P&L":float(p["net_pnl"]) if closed else p["btc_amount"]*fc-p["order_size_usdt"],
            "Holding Time":(p["sell_time"] if closed else ft)-p["buy_time"]})
    return pd.DataFrame(rows)


In [ ]:
data=load_data(); result=run_v0(data); v0=perf(data,result["equity_curve"]); result["equity_curve"]=v0["equity_curve"]
dca=monthly_dca(data); trades=history(result)
open_n=int(trades.Status.eq("OPEN").sum()); unreal=float(trades.loc[trades.Status.eq("OPEN"),"Net P&L"].sum())
summary={"initial_capital":INITIAL_CAPITAL,"initial_market_price":result["initialization"]["start_price"],
 "normal_order_size_usdt":result["initialization"]["normal_order_size_usdt"],"initial_cash":result["initialization"]["initial_cash"],
 "initial_btc":result["initialization"]["initial_btc"],"initial_sell_positions":result["initialization"]["initial_sell_positions"],
 "initial_buy_levels":result["initialization"]["initial_buy_levels"],"final_equity":v0["final_equity"],"net_return":v0["net_return"],
 "annualized_return":v0["annualized_return"],"max_drawdown":v0["max_drawdown"],"calmar_ratio":v0["calmar_ratio"],
 "completed_cycles":result["completed_cycles"],"open_positions":open_n,"final_cash":result["final_cash"],"final_btc":result["final_btc"],
 "realized_profit":result["realized_profit"],"unrealized_pnl":unreal,
 "total_fee_usdt_equiv":result["total_buy_fee_usdt"]+result["total_sell_fee_usdt"]}
dca_summary={k:dca[k] for k in ["number_of_buys","dca_amount_usdt","total_invested_usdt","total_buy_fee_usdt_equiv",
 "final_cash","final_btc","first_buy_time","last_buy_time","final_equity","net_return","annualized_return","max_drawdown","calmar_ratio"]}
cmp={"final_equity_difference_usdt":summary["final_equity"]-dca_summary["final_equity"],
     "excess_return":summary["net_return"]-dca_summary["net_return"],
     "drawdown_improvement":abs(dca_summary["max_drawdown"])-abs(summary["max_drawdown"]),
     "calmar_difference":summary["calmar_ratio"]-dca_summary["calmar_ratio"]}

ev=result["events"]; cashmove=float(ev.cash_movement.sum()); gridflow=float(ev.grid_cashflow.sum())
open_btc=sum(p["btc_amount"] for p in result["positions"].values() if p["status"]=="OPEN")
audit={
 "cash_reconciliation":bool(np.isclose(INITIAL_CAPITAL+cashmove,result["final_cash"],atol=1e-8)),
 "realized_profit_reconciliation":bool(np.isclose(gridflow,result["realized_profit"],atol=1e-8)),
 "cash_never_negative":bool(result["equity_curve"].cash.min()>=-1e-8),
 "initial_allocation_reconciliation":bool(np.isclose(result["initialization"]["initial_cash"]+result["initialization"]["initial_btc_cost_usdt"],INITIAL_CAPITAL,atol=1e-8)),
 "final_btc_matches_open_positions":bool(np.isclose(result["final_btc"],open_btc,atol=1e-10)),
 "closed_trade_count_reconciliation":bool(int(trades.Status.eq("CLOSED").sum())==result["completed_cycles"]),
 "dca_total_invested_matches_initial_capital":bool(np.isclose(dca["total_invested_usdt"],INITIAL_CAPITAL,atol=1e-8)),
 "dca_final_cash_zero":bool(np.isclose(dca["final_cash"],0,atol=1e-8)),
 "dca_final_equity_identity":bool(np.isclose(dca["final_equity"],dca["final_btc"]*float(data.iloc[-1].close)+dca["final_cash"],atol=1e-8))}
AUDIT_STATUS="PASS" if all(audit.values()) else "FAIL"

display(pd.DataFrame({"Metric":["Final Portfolio Value (USDT)","Net Return","Annualized Return","Max Drawdown","Calmar Ratio"],
 "V0-B Grid":[summary["final_equity"],summary["net_return"],summary["annualized_return"],summary["max_drawdown"],summary["calmar_ratio"]],
 "Monthly DCA":[dca_summary["final_equity"],dca_summary["net_return"],dca_summary["annualized_return"],dca_summary["max_drawdown"],dca_summary["calmar_ratio"]]}))
print("\nV0-B"); [print(k,":",v) for k,v in summary.items()]
print("\nMonthly DCA"); [print(k,":",v) for k,v in dca_summary.items()]
print("\nComparison vs DCA"); [print(k,":",v) for k,v in cmp.items()]
print("\nDCA Schedule"); display(dca["schedule"])
print("\nAudit",AUDIT_STATUS); [print(k,":",v) for k,v in audit.items()]
if AUDIT_STATUS!="PASS": raise AssertionError("V0-B AUDIT FAILED")


# 3. Trade History + Immutable Logging


In [ ]:
display(trades)

def js(v):
    if isinstance(v,dict): return {str(k):js(x) for k,x in v.items()}
    if isinstance(v,(list,tuple)): return [js(x) for x in v]
    if isinstance(v,np.ndarray): return [js(x) for x in v.tolist()]
    if isinstance(v,(np.integer,)): return int(v)
    if isinstance(v,(np.floating,float)): return float(v) if np.isfinite(v) else None
    if isinstance(v,(pd.Timestamp,datetime)): return None if pd.isna(v) else v.isoformat()
    if isinstance(v,(np.bool_,bool)): return bool(v)
    if v is pd.NaT or v is pd.NA: return None
    return v

run_id=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"); run_dir=f"logs/v0/{run_id}"
summary_path=f"{run_dir}/summary.json"; history_path=f"{run_dir}/trade_history.csv"
payload=js({"log_schema_version":6,"strategy":"V0-B Initialized Fixed Grid (Grid-Consistent Seed Sizing)",
 "run_info":{"run_id":run_id,"generated_at_utc":datetime.now(timezone.utc).isoformat(),"repository":"natdanaiii/Trading",
             "branch":"main","notebook":"Grid_trading_V0.ipynb","symbol":SYMBOL,"timeframe":TIMEFRAME,"start_date":START_DATE,
             "end_date":END_DATE,"data_rows":len(data),"data_first_time":data.open_time.min(),"data_last_time":data.open_time.max()},
 "trading_config":{"initial_capital":INITIAL_CAPITAL,"floor":GRID_FLOOR,"ceiling":GRID_CEILING,"gap":GRID_GAP,"buy_fee":BUY_FEE,"sell_fee":SELL_FEE},
 "initialization":result["initialization"],
 "market_range_diagnostics":{"historical_low":float(data.low.min()),"historical_high":float(data.high.max()),
   "candles_low_below_floor":int((data.low<GRID_FLOOR).sum()),"candles_high_above_ceiling":int((data.high>GRID_CEILING).sum())},
 "summary":summary,"monthly_dca_benchmark":dca_summary,"comparison_vs_monthly_dca":cmp,
 "audit":{"status":AUDIT_STATUS,"checks":audit},"trade_history_file":history_path})

def upload(path,text,msg):
    token=userdata.get("GITHUB_TOKEN"); url=f"https://api.github.com/repos/natdanaiii/Trading/contents/{path}"
    headers={"Authorization":f"Bearer {token}","Accept":"application/vnd.github+json","X-GitHub-Api-Version":"2022-11-28"}
    if requests.get(url,headers=headers,params={"ref":"main"},timeout=30).status_code==200: raise FileExistsError(path)
    body={"message":msg,"content":base64.b64encode(text.encode()).decode(),"branch":"main"}
    for a in range(1,4):
        r=requests.put(url,headers=headers,json=body,timeout=30)
        if r.status_code in (200,201): return r.json()["commit"]["sha"]
        if r.status_code==409 and a<3: time.sleep(a); continue
        r.raise_for_status()

print("Run ID:",run_id)
print("Summary upload:",upload(summary_path,json.dumps(payload,indent=2,allow_nan=False),f"Add V0-B summary {run_id}"))
print("History upload:",upload(history_path,trades.to_csv(index=False),f"Add V0-B trade history {run_id}"))
